# Institution resource-list cleanup

Post-processes the output of the `accredited-institution-resources` skill. Two independent
operations, each reading an existing xlsx and writing a **new** file next to it — neither
operation ever overwrites its input. See `../SKILL.md` for the full rationale behind each.

Both operations expect at least these columns: `unitid, institution_name, homepage_url, city,
state, accreditor, pct_biomedical, has_biomedical_program_data, base_domain, resource_list_url,
resource_list_notes`.

## Operation 1: tighten the religious-institution filter

Adds two signals beyond what `accredited-institution-resources`'s Filter 5 already checks
(faith-related accreditor name, theology-program share):

- **Name-pattern match** — catches institutions whose `accreditor` field in the Scorecard API only
  lists a regional accreditor even though they're also accredited by a faith-related one in reality
  (e.g. Southeastern Baptist Theological Seminary is ATS-accredited, but the API only returns
  SACSCOC for it).
- **Manual override list** (`../reference/manual_religious_institution_overrides.csv`) — a small,
  curated list of institutions with no religious keyword in their name at all (Regent University,
  Calvin University, Spertus College, Columbia International University), verified religious by
  other means. Kept as an explicit list rather than widening the regex, to avoid false-positiving
  on secular institutions with similar-sounding names.

In [ ]:
import pathlib
import re

import pandas as pd

# Point this at whatever spreadsheet needs the tightened religious filter applied.
INPUT_PATH = pathlib.Path("../../output/accredited_nonprofit_secular_institutions.xlsx")

df = pd.read_excel(INPUT_PATH)

RELIGIOUS_NAME_PATTERN = re.compile(
    r"\bseminary\b|\byeshiva\b|\btalmudic\b|\brabbinical\b|\btorah\b|"
    r"jewish institute of religion|bible institute|bible college|biblical institute|"
    r"biblical seminary|school of theology|theological school|theological seminary|"
    r"divinity school|school of divinity|\btheological\b",
    re.IGNORECASE,
)

overrides = pd.read_csv("../reference/manual_religious_institution_overrides.csv")
override_unitids = set(overrides["unitid"])

name_match = df["institution_name"].str.contains(RELIGIOUS_NAME_PATTERN, na=False)
override_match = df["unitid"].isin(override_unitids)
remove_mask = name_match | override_match

removed = df[remove_mask]
print(f"{len(removed)} rows matched ({int(name_match.sum())} by name pattern, "
      f"{int(override_match.sum())} by manual override) across "
      f"{removed['base_domain'].nunique()} domains:")
for idx, row in removed.iterrows():
    reason = "name pattern" if name_match[idx] else "manual override"
    print(f"  {row['unitid']} | {row['institution_name']} | {row['base_domain']} | {reason}")

filtered = df[~remove_mask].copy()

FILTERED_OUTPUT_PATH = INPUT_PATH.with_name(INPUT_PATH.stem + "_religious_filtered" + INPUT_PATH.suffix)
filtered.to_excel(FILTERED_OUTPUT_PATH, index=False)

print()
print(f"Input:  {len(df)} rows ({INPUT_PATH})")
print(f"Output: {len(filtered)} rows ({FILTERED_OUTPUT_PATH})")

## Operation 2: de-duplicate by resource URL

Many institutions share the exact same `resource_list_url` — not just branch campuses of the same
system (already grouped by `base_domain` upstream), but also different `base_domain`s that
independently fell back to the same shared page (e.g. several Penn State branch-campus domains all
fall back to the same main Penn State Libraries page). De-duplicating by `base_domain` alone doesn't
collapse these; de-duplicating by `resource_list_url` does.

Output keeps the same columns, one row per unique `resource_list_url`. Where multiple institutions
share a URL, their identifying fields become a `; `-joined list of every institution using that URL
— nothing is silently dropped. `pct_biomedical` takes the max across the group (preserving the
upstream sort/priority behavior) and `has_biomedical_program_data` is true if any member has it.

`INPUT_PATH_2` defaults to the same base spreadsheet as Operation 1 (independent of it) — point it
at `FILTERED_OUTPUT_PATH` instead if you want to run the two operations chained.

In [ ]:
import pathlib

import pandas as pd

INPUT_PATH_2 = pathlib.Path("../../output/accredited_nonprofit_secular_institutions.xlsx")

COLS = [
    "unitid", "institution_name", "homepage_url", "city", "state", "accreditor",
    "pct_biomedical", "has_biomedical_program_data", "base_domain",
    "resource_list_url", "resource_list_notes",
]

df2 = pd.read_excel(INPUT_PATH_2)


def join_unique(series):
    seen = []
    for v in series:
        s = "" if pd.isna(v) else str(v)
        if s not in seen:
            seen.append(s)
    return "; ".join(seen)


grouped = df2.groupby("resource_list_url", sort=False).agg(
    unitid=("unitid", join_unique),
    institution_name=("institution_name", join_unique),
    homepage_url=("homepage_url", join_unique),
    city=("city", join_unique),
    state=("state", join_unique),
    accreditor=("accreditor", join_unique),
    pct_biomedical=("pct_biomedical", "max"),
    has_biomedical_program_data=("has_biomedical_program_data", "any"),
    base_domain=("base_domain", join_unique),
    resource_list_notes=("resource_list_notes", join_unique),
).reset_index()

grouped = grouped[COLS]
grouped = grouped.sort_values(["pct_biomedical", "institution_name"], ascending=[False, True]).reset_index(drop=True)

DEDUPED_OUTPUT_PATH = INPUT_PATH_2.with_name(INPUT_PATH_2.stem + "_unique_resource_urls" + INPUT_PATH_2.suffix)
grouped.to_excel(DEDUPED_OUTPUT_PATH, index=False)

n_merged_groups = int((df2.groupby("resource_list_url").size() > 1).sum())
print(f"Input:  {len(df2)} rows, {df2['resource_list_url'].nunique()} unique resource_list_url values ({INPUT_PATH_2})")
print(f"Output: {len(grouped)} rows, one per unique resource_list_url ({DEDUPED_OUTPUT_PATH})")
print(f"{n_merged_groups} groups had 2+ institutions merged into one row")